# A* (A星) 算法：网格寻路

本笔记本包含：
1. A* 核心实现（4邻接/可选8邻接）
2. 网格示例（障碍、起点、终点）
3. 输出最短路径与可视化（字符画）


In [24]:
from __future__ import annotations

import heapq
from dataclasses import dataclass
from typing import Callable, Dict, Iterable, List, Optional, Tuple

Grid = List[List[int]]  # 0: free, 1: obstacle
Pos = Tuple[int, int]     # (row, col)


In [25]:
@dataclass(frozen=True)
class Node:
    pos: Pos
    g: float
    f: float


def heuristic_manhattan(a: Pos, b: Pos) -> int:
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def heuristic_euclidean(a: Pos, b: Pos) -> float:
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5


In [26]:
def reconstruct_path(came_from: Dict[Pos, Pos], current: Pos) -> List[Pos]:
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path


In [27]:
def astar(
    grid: Grid,
    start: Pos,
    goal: Pos,
    *,
    heuristic: Callable[[Pos, Pos], float] = heuristic_manhattan,
    allow_diagonal: bool = False,
) -> Optional[List[Pos]]:
    """A* 网格寻路。

    - grid: 0可走，1不可走
    - start/goal: (row, col)
    - allow_diagonal: 是否允许对角移动

    返回：从 start 到 goal 的路径（包含起点和终点），找不到则返回 None
    """
    rows = len(grid)
    cols = len(grid[0]) if rows else 0

    def in_bounds(p: Pos) -> bool:
        r, c = p
        return 0 <= r < rows and 0 <= c < cols

    def is_walkable(p: Pos) -> bool:
        r, c = p
        return grid[r][c] == 0

    if not in_bounds(start) or not in_bounds(goal):
        raise ValueError("start/goal 超出网格范围")
    if not is_walkable(start) or not is_walkable(goal):
        raise ValueError("start/goal 在障碍上")

    if start == goal:
        return [start]

    if allow_diagonal:
        directions = [
            (-1, 0), (1, 0), (0, -1), (0, 1),
            (-1, -1), (-1, 1), (1, -1), (1, 1),
        ]
    else:
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    # g_score: 已找到的最优代价
    g_score: Dict[Pos, float] = {start: 0.0}
    came_from: Dict[Pos, Pos] = {}

    # 允许重复入堆；通过 g_score 来过滤“过期节点”
    open_heap: List[Tuple[float, int, Node]] = []
    counter = 0

    start_h = float(heuristic(start, goal))
    heapq.heappush(open_heap, (start_h, counter, Node(start, 0.0, start_h)))

    closed: set[Pos] = set()

    while open_heap:
        _, _, current_node = heapq.heappop(open_heap)
        current = current_node.pos

        if current in closed:
            continue
        closed.add(current)

        if current == goal:
            return reconstruct_path(came_from, current)

        cur_g = g_score[current]

        for dr, dc in directions:
            nr, nc = current[0] + dr, current[1] + dc
            neighbor = (nr, nc)
            if not in_bounds(neighbor) or not is_walkable(neighbor):
                continue

            # 边权：对角移动稍贵（欧氏距离近似）
            step_cost = 1.0
            if dr != 0 and dc != 0:
                step_cost = 2 ** 0.5

            tentative_g = cur_g + step_cost

            if neighbor in closed and tentative_g >= g_score.get(neighbor, float("inf")):
                continue

            if tentative_g < g_score.get(neighbor, float("inf")):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f = tentative_g + float(heuristic(neighbor, goal))
                counter += 1
                heapq.heappush(open_heap, (f, counter, Node(neighbor, tentative_g, f)))

    return None


In [28]:
def draw_path(grid: Grid, path: Optional[List[Pos]], start: Pos, goal: Pos) -> str:
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    path_set = set(path) if path else set()
    lines: List[str] = []
    for r in range(rows):
        row_chars = []
        for c in range(cols):
            p = (r, c)
            if p == start:
                row_chars.append('S')
            elif p == goal:
                row_chars.append('G')
            elif grid[r][c] == 1:
                row_chars.append('#')
            elif p in path_set:
                row_chars.append('*')
            else:
                row_chars.append('.')
        lines.append(' '.join(row_chars))
    return "\n".join(lines)


In [ ]:
# =============== 示例：你可以改这里的网格/起点终点 ===============
# 0: 可走, 1: 障碍
grid: Grid = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 1, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
]

start: Pos = (0, 0)
goal: Pos = (8, 9)

# 对 4 邻接通常用曼哈顿；对 8 邻接可用欧氏
path = astar(
    grid,
    start,
    goal,
    heuristic=heuristic_manhattan,
    allow_diagonal=False,
)

print("是否找到路径:", path is not None)
if path:
    print("路径长度(步数, 含起终点):", len(path))
    print("路径坐标:", path)
print("\n路径可视化(字符画):")
print(draw_path(grid, path, start, goal))


是否找到路径: True
路径长度(步数, 含起终点): 18
路径坐标: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (8, 6), (8, 7), (8, 8), (8, 9)]

路径可视化(字符画):
S . . . . . . . . .
* # # # . # # # # .
* . . # . . . . # .
* # . # # # # . # .
* # . . . . # . . .
* # # # # . # # # .
* . . . # . . . . .
* # # . # # # # # .
* * * * * * * * * G


: 